connection of google drive

In [ ]:
!rm -rf /content/dataset/Bijie-landslide-dataset

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
!mkdir -p /content/dataset/biji_dataset
!cp -r "/content/drive/MyDrive/Bijie-landslide-dataset/"* "/content/dataset/biji_dataset"

In [16]:
import os

DATA_ROOT = "/content/dataset/biji_dataset"

print("Exists:", os.path.exists(DATA_ROOT))

print("\nLandslide images:",
      len(os.listdir(os.path.join(DATA_ROOT, "landslide/image"))))

print("Landslide masks:",
      len(os.listdir(os.path.join(DATA_ROOT, "landslide/mask"))))

print("Non-landslide images:",
      len(os.listdir(os.path.join(DATA_ROOT, "non-landslide/image"))))

Exists: True

Landslide images: 770
Landslide masks: 770
Non-landslide images: 2027


In [15]:
!ls -R /content/dataset/biji_dataset

/content/dataset/biji_dataset:
landslide  non-landslide

/content/dataset/biji_dataset/landslide:
dem  image  mask  polygon_coordinate

/content/dataset/biji_dataset/landslide/dem:
df002.png  hz076.png  ny009.png  qx047.png  qx157.png	qxg099.png  zj013.png
df003.png  hz077.png  ny010.png  qx048.png  qx158.png	qxg100.png  zj014.png
df004.png  hz078.png  ny011.png  qx049.png  qx159.png	qxg101.png  zj015.png
df005.png  hz079.png  ny012.png  qx050.png  qx160.png	qxg102.png  zj016.png
df006.png  hz080.png  ny013.png  qx051.png  qx161.png	qxg103.png  zj017.png
df007.png  hz081.png  ny014.png  qx052.png  qx162.png	qxg104.png  zj018.png
df008.png  hz082.png  ny015.png  qx053.png  qx163.png	qxg105.png  zj019.png
df009.png  hz083.png  ny016.png  qx054.png  qx164.png	qxg106.png  zj020.png
df010.png  hz084.png  ny017.png  qx055.png  qx165.png	qxg107.png  zj021.png
df011.png  hz085.png  ny018.png  qx056.png  qx166.png	qxg108.png  zj022.png
df012.png  hz086.png  ny019.png  qx057.png  qx167.png	qxg10

Importing the zip file of dataset

In [32]:
# Set dataset path (change this if your dataset is somewhere else)
DATA_ROOT = "/content/dataset/biji_dataset"

# Required Libraries

In [19]:
!pip install torch torchvision tqdm scikit-learn pillow

# Import Libraries

In [33]:
import os
import glob
from PIL import Image
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF

from sklearn.metrics import precision_recall_fscore_support, accuracy_score


# Dataset Class for Google Colab
Handles:
1.Landslide images + masks
2.Non-landslide images → auto-generate zero masks
3.Resize
4.Normalize

In [34]:
IMG_SIZE = (256, 256)

class BijiSegmentationDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.transforms = transforms

        self.ls_img = sorted(glob.glob(os.path.join(root_dir, "landslide/image/*")))
        self.ls_mask = sorted(glob.glob(os.path.join(root_dir, "landslide/mask/*")))
        self.nonls_img = sorted(glob.glob(os.path.join(root_dir, "non-landslide/image/*")))

        self.samples = []
        for img in self.ls_img:
            base = os.path.splitext(os.path.basename(img))[0]
            mask = os.path.join(root_dir, "landslide/mask", base + ".png")
            if not os.path.exists(mask):
                mask = os.path.join(root_dir, "landslide/mask", base + ".jpg")
            self.samples.append((img, mask))

        for img in self.nonls_img:
            self.samples.append((img, None))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img = Image.open(img_path).convert("RGB")
        if mask_path is None:
            mask = Image.fromarray(np.zeros((img.height, img.width), dtype=np.uint8))
        else:
            mask = Image.open(mask_path).convert("L")

        img = img.resize(IMG_SIZE, Image.BILINEAR)
        mask = mask.resize(IMG_SIZE, Image.NEAREST)

        img = TF.to_tensor(img)
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])

        mask = torch.tensor(np.array(mask) > 127, dtype=torch.float32).unsqueeze(0)

        return img, mask

# U-Net Architecture

In [35]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.seq(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch)
        )

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        x1 = nn.functional.pad(x1, [diffX//2, diffX-diffX//2,
                                    diffY//2, diffY-diffY//2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.inc = DoubleConv(in_ch, 32)
        self.down1 = Down(32, 64)
        self.down2 = Down(64, 128)
        self.down3 = Down(128, 256)
        self.down4 = Down(256, 256)
        self.up1 = Up(512, 128)
        self.up2 = Up(256, 64)
        self.up3 = Up(128, 32)
        self.up4 = Up(64, 32)
        self.outc = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)


# Loss Function

In [37]:
def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    pred_f = pred.reshape(-1)
    targ_f = target.reshape(-1)
    inter = (pred_f * targ_f).sum()
    return 1 - ((2*inter + smooth) / (pred_f.sum() + targ_f.sum() + smooth))

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, target):
        return 0.5*self.bce(logits, target) + 0.5*dice_loss(logits, target)


# Training Setup

In [38]:
dataset = BijiSegmentationDataset(DATA_ROOT)
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = BCEDiceLoss()

# Training Loop

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

EPOCHS = 30

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        logits = model(imgs)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    accs, ps, rs, f1s = [], [], [], []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)

            loss = criterion(logits, masks)
            val_loss += loss.item()

            preds = (torch.sigmoid(logits) > 0.5).float()

            accs.append(accuracy_score(masks.cpu().numpy().flatten(),
                                       preds.cpu().numpy().flatten()))

            p, r, f1, _ = precision_recall_fscore_support(
                masks.cpu().numpy().flatten(),
                preds.cpu().numpy().flatten(),
                average="binary",
                zero_division=0
            )
            ps.append(p); rs.append(r); f1s.append(f1)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Acc: {np.mean(accs):.4f} | P: {np.mean(ps):.4f} | "
          f"R: {np.mean(rs):.4f} | F1: {np.mean(f1s):.4f}")


# Save the Trained Model

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/unet_landslide_best.pth")
print("Model saved!")
